# FastText POC — Eval vs LLM

Run after `prepare.py → train.py`. Compares fastText to LLM outputs (from `categorizations` table) and shows confusion, low-conf fallback, and held-out merchant generalization.


In [ ]:
import pathlib, json, csv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

ROOT = pathlib.Path("../data/processed")
with open(ROOT / "metrics_summary.json") as f:
    m=json.load(f)
print(json.dumps({k: (v if not isinstance(v, dict) else {ik: round(iv,3) if isinstance(iv,float) else iv for ik,iv in v.items() if ik in ("accuracy","macro_f1","low_conf_rate","n")}) for k,v in m.items() if k!="latency_ms"}, indent=2))
print("latency", m.get("latency_ms"))

In [ ]:
# Confusion matrix for test
import numpy as np
d = m.get("test")
if d:
    labels=d["labels"]
    cm=np.array(d["confusion"])
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
    plt.title("Confusion — test")
    plt.show()
else:
    print("no test metrics yet")

In [ ]:
# Held-out merchant gap
for k in ["test","heldout_merchant"]:
    if k in m:
        print(k, f"acc={m[k]['accuracy']:.3f} mf1={m[k]['macro_f1']:.3f} low={m[k]['low_conf_rate']:.1%}")
if "test" in m and "heldout_merchant" in m:
    print("gap mf1", m["test"]["macro_f1"]-m["heldout_merchant"]["macro_f1"])

In [ ]:
# Live predict demo
import sys; sys.path.insert(0, "..")
try:
    import fasttext
    from normalize import merchant_for_training
    model=fasttext.load_model(str(pathlib.Path("../models/merchant_ft.bin")))
    for raw in ["TIM HORTONS #3356       BURNABY","UBER EATS               TORONTO","NETFLIX.COM             VANCOVER","BC-HYDRO-BILL-PMNT      800-224-9376","PAYMENT RECEIVED - THANK YOU"]:
        txt=merchant_for_training(raw, -1200)
        pred,prob=model.predict(txt, k=2)
        print(f"{raw:40} -> {pred} {prob} | txt={txt!r}")
except Exception as e:
    print(e)

## Next: compare to LLM
Export LLM labels: `SELECT merchant_raw, category_id, ai_confidence FROM transactions WHERE category_id IS NOT NULL LIMIT 500` → join to same test merchants, compute agreement rate and where fastText low-conf <0.6 matches LLM low-conf.